# Script to run surface simulation and collect the fluxes

In [1]:
import os
import re
import cantera as ct
import numpy as np
import matplotlib.pyplot as plt


In [2]:
yaml_file = 'chem_annotated.yaml'

In [3]:
gas, _ = ct.import_phases(yaml_file, ["gas", "surface1"])
surf = ct.Interface(yaml_file, 'surface1')

/home/moon/anaconda3/envs/rmg_env/lib/python3.11/site-packages/cantera/utils.py:22: UserWarning: StickingRate::validate: 
Sticking coefficient is greater than 1 for reaction 'N2(10) + 2 X(1) <=> N2X2(29)'
at T = 200.0
at T = 500.0
at T = 1000.0
at T = 2000.0
at T = 5000.0
at T = 10000.0

  return [Solution(filename, p) for p in phase_names]
/tmp/ipykernel_225/328003026.py:2: UserWarning: StickingRate::validate: 
Sticking coefficient is greater than 1 for reaction 'N2(10) + 2 X(1) <=> N2X2(29)'
at T = 200.0
at T = 500.0
at T = 1000.0
at T = 2000.0
at T = 5000.0
at T = 10000.0

  surf = ct.Interface(yaml_file, 'surface1')


In [6]:
# Get O2 and NH3 indices
for i in range(gas.n_species):
    if gas.species()[i].composition == {'O': 2.0}:
        i_O2 = i
        break
else:
    raise ValueError('No O2 found!')
    
for i in range(gas.n_species):
    if gas.species()[i].composition == {'N': 1.0, 'H': 3.0}:
        i_NH3 = i
        break
else:
    raise ValueError('No NH3 found!')

# Run a simulation

In [9]:
T = 673.15  # 400 C
P = ct.one_atm
initial_composition = f'{gas.species()[i_O2].name}: {0.05}, {gas.species()[i_NH3].name}: {0.95}'

infinite_flow = True


t_max = 1e3
# import the model and set the initial conditions
surf = ct.Interface(yaml_file, 'surface1')
surf.TP = T, P
initial_coverages = np.zeros_like(surf.coverages)
initial_coverages[0] = 1.0
surf.coverages = initial_coverages  # initial condition: all sites are vacent
gas = surf.adjacent['gas']
gas.TPX = T, P, initial_composition
time = 0.0

    
r = ct.IdealGasReactor(gas, energy='off')
r.volume = 6.0

# # create a reservoir to represent the reactor immediately upstream. Note
# # that the gas object is set already to the state of the upstream reactor
upstream = ct.Reservoir(gas, name='upstream')

# # create a reservoir for the reactor to exhaust into. The composition of
# # this reservoir is irrelevant.
downstream = ct.Reservoir(gas, name='downstream')

cov = surf.coverages

# Add the reacting surface to the reactor. The area is set to the desired
# catalyst area in the reactor.
rsurf = ct.ReactorSurface(surf, r, A=24.0)

if infinite_flow:
    V = 1e30
else:
    V = 30 # velocity m/s (setting very high values allows for effectively fixed gas-phase composition)

mass_flow_rate = V * gas.density * 1 * 1
# # The mass flow rate into the reactor will be fixed by using a
# # MassFlowController object.
# m = ct.MassFlowController(upstream, r, mdot=mass_flow_rate)
m = ct.MassFlowController(upstream, r, mdot=mass_flow_rate)

# # We need an outlet to the downstream reservoir. This will determine the
# # pressure in the reactor. The value of K will only affect the transient
# # pressure difference.
v = ct.PressureController(r, downstream, primary=m, K=1e-5)  # cantera >= 2.6

sim = ct.ReactorNet([r])   
# sim.atol = 1e-18
# sim.rtol = 1e-15

# Create a SolutionArray to store the data
time_history = ct.SolutionArray(gas, extra=["time", "surf_coverages", "surf_net_rate_progress", "surf_net_prod"])
    
# MAX_STEPS = 1e5
while time < t_max:

    time = sim.step()
    time_history.append(r.thermo.state, time=time, surf_coverages=rsurf.kinetics.coverages, surf_net_rate_progress=rsurf.kinetics.net_rates_of_progress,
                       surf_net_prod=rsurf.kinetics.net_production_rates)
    # if len(time_history.time) > MAX_STEPS:
    #     break

/tmp/ipykernel_225/453107860.py:10: UserWarning: StickingRate::validate: 
Sticking coefficient is greater than 1 for reaction 'N2(10) + 2 X(1) <=> N2X2(29)'
at T = 200.0
at T = 500.0
at T = 1000.0
at T = 2000.0
at T = 5000.0
at T = 10000.0

  surf = ct.Interface(yaml_file, 'surface1')
/tmp/ipykernel_225/453107860.py:20: DeprecationWarning: ReactorBase.__init__: After Cantera 3.2, the default value of the `clone` argument will be `True`, resulting in an independent copy of the `phase` being created for use by this reactor. Add the `clone=False` argument to retain the old behavior of sharing `Solution` objects.
  r = ct.IdealGasReactor(gas, energy='off')
/tmp/ipykernel_225/453107860.py:25: DeprecationWarning: ReactorBase.__init__: After Cantera 3.2, the default value of the `clone` argument will be `True`, resulting in an independent copy of the `phase` being created for use by this reactor. Add the `clone=False` argument to retain the old behavior of sharing `Solution` objects.
  upstre

# Save the time steps, gas/surf cocentrations, and rates of progress

In [10]:
times_file = os.path.join(os.path.dirname(yaml_file), 'sim_times.npy')
np.save(times_file, time_history.time)

gas_concs_file = os.path.join(os.path.dirname(yaml_file), 'gas_concs.npy')
np.save(gas_concs_file, time_history.concentrations)

surf_covs_file = os.path.join(os.path.dirname(yaml_file), 'surf_covs.npy')
np.save(surf_covs_file, time_history.surf_coverages)

gas_rates_file = os.path.join(os.path.dirname(yaml_file), 'gas_rates.npy')
np.save(gas_rates_file, time_history.net_rates_of_progress)

surf_rates_file = os.path.join(os.path.dirname(yaml_file), 'surf_rates.npy')
np.save(surf_rates_file, time_history.surf_net_rate_progress)

